# 🎙️ OmniASR-CTC-1B — Tunisian Arabic ASR Benchmark

**Model:** facebook/omniASR-CTC-1B  
**Config key:** omniasr_ctc_1b

> OmniASR uses the omnilingual-asr package (fairseq2-based), NOT HF Transformers.  
> Inference goes through ASRInferencePipeline with Arabic language tag ara_Arab.  
> Audio is passed as [{'waveform': array, 'sample_rate': sr}] dicts.  
> Constraint: segments must be ≤ 40 s; longer audio must be pre-chunked.

## 1 · Install & Imports

In [ ]:
!pip install -q omnilingual-asr datasets evaluate jiwer pyyaml
# fairseq2 requires libsndfile on Colab:
# !apt-get install -y -q libsndfile1

Found existing installation: omnilingual-asr 0.1.0
Uninstalling omnilingual-asr-0.1.0:
  Successfully uninstalled omnilingual-asr-0.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.7/512.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 757.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.3 MB/s

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Clone / upload the benchmarking package to Colab ─────────────────────
!cp -r '/content/drive/My Drive/asr_benchmark' /content/asr_benchmark
import sys
sys.path.insert(0, '/content/asr_benchmark')

In [ ]:
from benchmark_utils import (
    load_config, get_device, print_gpu_info, setup_output_dir,
    load_benchmark, split_benchmark,
    compute_metrics, per_sample_wer,
    run_pipeline_inference, build_results_df,
    run_labelled_splits, run_unlabelled_splits,
    display_preview, display_worst, display_bulk_predictions,
    audio_inspector, display_summary, plot_wer_cer,
)
import numpy as np
import torch
import time
from tqdm.auto import tqdm
from datasets import Audio as HFAudio

cfg = load_config('/content/asr_benchmark/config.yaml')
TARGET_SR    = cfg['evaluation']['target_sr']
TOP_N_WORST  = cfg['evaluation']['top_n_worst']
PREVIEW_ROWS = cfg['evaluation']['preview_rows']

## 2 · GPU Check

In [ ]:
device = get_device()
print_gpu_info()

Device : cpu
⚠️  No GPU found — inference will be slow on CPU.


## 3 · Load Model & Pipeline

### 3.1 Load `ASRInferencePipeline` (`omniASR_CTC_1B`)
`ASRInferencePipeline` downloads and caches the model weights on first call.
Pass `model_card` exactly as listed in the omnilingual-asr registry.

| Parameter | Value | Note |
|-----------|-------|--------|
| `model_card` | `omniASR_CTC_1B` | Registry name |  
| `language` | `ara_Arab` | BCP-47 + script |  
| `batch_size` | from config | Tune to GPU VRAM |



In [ ]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

mcfg        = cfg['models']['omniasr_ctc_1b']
OUTPUT_DIR  = setup_output_dir(cfg, 'omniasr_ctc_1b')
FILE_SUFFIX = 'omniasr_ctc_1b_results'
BATCH_SIZE  = mcfg['batch_size']
LANG        = mcfg['language']

print("Loading omniASR_CTC_1B ...")
omni_pipe = ASRInferencePipeline(model_card=mcfg['model_card'])
print("✓ omniASR_CTC_1B loaded")

RuntimeError: fairseq2 requires a CUDA 12.4 build of PyTorch 2.6.0, but the installed version is a CPU-only build of PyTorch 2.10.0. Either follow the instructions at https://pytorch.org/get-started/locally to update PyTorch, or the instructions at https://github.com/facebookresearch/fairseq2#variants to update fairseq2.

## 4 · Mount Drive & Load Benchmark

In [ ]:
benchmark = load_benchmark(cfg)
LABELLED_SPLITS, UNLABELLED_SPLITS = split_benchmark(benchmark)

## 5 · Inference

In [ ]:
RESUME = False

def infer_fn(ds):
    """
    Wraps ASRInferencePipeline.transcribe().
    Input format: [{'waveform': np.ndarray, 'sample_rate': int}]
    Language tag: 'ara_Arab'.
    Note: OmniASR only accepts segments <= 40 s.
    """
    ds = ds.cast_column('audio', HFAudio(sampling_rate=TARGET_SR))
    predictions, latencies = [], []
    start = time.time()

    for i in tqdm(range(0, len(ds), BATCH_SIZE), desc='Inferring', unit='batch'):
        batch = ds.select(range(i, min(i + BATCH_SIZE, len(ds))))

        audio_data = [
            {
                'waveform': np.array(x['audio']['array'], dtype=np.float32),
                'sample_rate': TARGET_SR
            }
            for x in batch
        ]

        t0 = time.time()
        texts = omni_pipe.transcribe(
            audio_data,
            lang=[LANG] * len(audio_data),
            batch_size=BATCH_SIZE,
        )

        elapsed = time.time() - t0
        per_s = elapsed / len(audio_data)

        for txt in texts:
            predictions.append(txt)
            latencies.append(per_s)

    return predictions, latencies, time.time() - start


all_result_dfs, summary_rows = run_labelled_splits(
    benchmark,
    LABELLED_SPLITS,
    infer_fn,
    OUTPUT_DIR,
    FILE_SUFFIX,
    PREVIEW_ROWS,
    TOP_N_WORST,
    resume=RESUME,
)

## 6 · Per-sample preview

In [ ]:
display_preview(all_result_dfs, PREVIEW_ROWS)

## 7 · Worst predictions

In [ ]:
display_worst(all_result_dfs, TOP_N_WORST)

## 8 · Unlabelled inspection

In [ ]:
unlabelled_result_dfs = run_unlabelled_splits(
    benchmark, UNLABELLED_SPLITS, infer_fn, OUTPUT_DIR, 'mms_results'
)

In [ ]:
unlabelled_result_dfs = run_unlabelled_splits(
    benchmark,
    UNLABELLED_SPLITS,
    infer_fn,
    OUTPUT_DIR,
    FILE_SUFFIX,
    resume=RESUME,
)

In [ ]:
display_bulk_predictions(unlabelled_result_dfs)

## 9 · Summary

In [ ]:
summary_df = display_summary(
    summary_rows, OUTPUT_DIR, 'mms', 'facebook/mms-1b-all'
)

In [ ]:
if summary_df is not None:
    plot_wer_cer(summary_df, OUTPUT_DIR, 'mms', 'facebook/mms-1b-all')